In [1]:
!pip install gymnasium
import numpy as np
import gymnasium as gym

env = gym.make("CliffWalking-v1")

print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)

gamma = 0.9
theta = 1e-10

def policy_evaluation(policy, value_table, env, gamma, theta):
    while True:
        delta = 0

        for state in range(env.observation_space.n):
            action = policy[state]
            v = 0

            for transition in env.unwrapped.P[state][action]:
                prob, next_state, reward, terminated = transition
                v += prob * (reward + gamma * value_table[next_state])

            delta = max(delta, abs(value_table[state] - v))
            value_table[state] = v

        if delta < theta:
            break

    return value_table

def policy_improvement(value_table, policy, env, gamma):
    policy_stable = True

    for state in range(env.observation_space.n):
        old_action = policy[state]
        action_values = np.zeros(env.action_space.n)

        for action in range(env.action_space.n):
            for transition in env.unwrapped.P[state][action]:
                prob, next_state, reward, terminated = transition
                action_values[action] += prob * (
                    reward + gamma * value_table[next_state]
                )

        policy[state] = np.argmax(action_values)

        if old_action != policy[state]:
            policy_stable = False

    return policy, policy_stable

def policy_iteration(env, gamma, theta):
    policy = np.random.choice(
        env.action_space.n,
        size=env.observation_space.n
    )

    value_table = np.zeros(env.observation_space.n)

    while True:
        value_table = policy_evaluation(
            policy,
            value_table,
            env,
            gamma,
            theta
        )

        policy, policy_stable = policy_improvement(
            value_table,
            policy,
            env,
            gamma
        )

        if policy_stable:
            return policy, value_table

optimal_policy, optimal_value = policy_iteration(
    env,
    gamma,
    theta
)

print("\nOptimal Policy:")
print(optimal_policy.reshape(4, 12))

print("\nOptimal Value Function:")
print(optimal_value.reshape(4, 12))

env.close()

/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment CliffWalking-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


DeprecatedEnv: Environment version v0 for `CliffWalking` is deprecated. Please use `CliffWalking-v1` instead.